# Audio Unimodal Emotion Recognition (MELD)

CNN-based emotion classifier on the MELD dataset using Log-Mel Spectrograms.

## 1. Imports & Config

In [1]:
import warnings
from collections import Counter
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm import tqdm

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT   = Path("../Data/MELD.Audio")
CSV_ROOT    = Path("../Data/MELD.Raw")
MODEL_DIR   = Path("../Models/Audio_Models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Audio config ─────────────────────────────────────────────────────────────
SAMPLE_RATE = 16_000
MAX_SECONDS = 6
MAX_LEN     = SAMPLE_RATE * MAX_SECONDS   # samples

# ── Label config ─────────────────────────────────────────────────────────────
EMO_MAP = {"joy": "happiness"}            # normalise dataset label names
LABELS  = ["anger", "disgust", "fear", "happiness", "sadness", "surprise", "neutral"]
NUM_CLASSES = len(LABELS)
LABEL2IDX   = {l: i for i, l in enumerate(LABELS)}

# ── Training config ──────────────────────────────────────────────────────────
BATCH_SIZE = 32
EPOCHS     = 50
LR         = 5e-4
PATIENCE   = 10          # early-stopping patience (epochs without F1 improvement)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## 2. Feature Extraction

In [2]:
# Log-Mel Spectrogram transform  (n_mels=64 avoids the all-zero filterbank warning
# that occurs when n_mels=128 with n_fft=400 → n_freqs=201)
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=400,
    hop_length=160,
    n_mels=64,
)
amplitude_to_db = torchaudio.transforms.AmplitudeToDB()


def fix_length(waveform: torch.Tensor) -> torch.Tensor:
    """Truncate or zero-pad waveform to exactly MAX_LEN samples. Shape: [1, T]."""
    T = waveform.shape[1]
    if T > MAX_LEN:
        return waveform[:, :MAX_LEN]
    if T < MAX_LEN:
        return F.pad(waveform, (0, MAX_LEN - T))
    return waveform


def load_audio(path) -> torch.Tensor:
    """Load audio file, resample to SAMPLE_RATE, and fix length. Returns [1, MAX_LEN]."""
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    waveform = torch.from_numpy(y).float().unsqueeze(0)
    return fix_length(waveform)


def extract_features(waveform: torch.Tensor) -> torch.Tensor:
    """Convert waveform [1, T] → log-mel spectrogram [n_mels, time]."""
    features = mel_transform(waveform)      # [1, n_mels, time]
    features = amplitude_to_db(features)
    return features.squeeze(0)              # [n_mels, time]

## 3. Dataset

In [3]:
CSV_PATHS = {
    "train": CSV_ROOT / "train" / "train_sent_emo.csv",
    "dev":   CSV_ROOT / "dev_sent_emo.csv",
    "test":  CSV_ROOT / "test_sent_emo.csv",
}


class MELDAudioDataset(Dataset):
    def __init__(self, split: str = "train"):
        assert split in CSV_PATHS, f"split must be one of {list(CSV_PATHS)}"
        self.audio_dir = DATA_ROOT / split

        df = pd.read_csv(CSV_PATHS[split])
        df["Emotion"] = df["Emotion"].str.lower().replace(EMO_MAP)

        # Build file paths vectorised (avoids slow iterrows)
        df["filepath"] = df.apply(
            lambda r: self.audio_dir / f"dia{r.Dialogue_ID}_utt{r.Utterance_ID}.wav",
            axis=1,
        )

        # Drop rows whose audio file is missing
        exists_mask = df["filepath"].apply(lambda p: p.exists())
        n_missing = (~exists_mask).sum()
        if n_missing:
            warnings.warn(f"[{split}] {n_missing} audio files not found — skipping.")
        df = df[exists_mask].reset_index(drop=True)

        # Drop unknown labels
        known_mask = df["Emotion"].isin(LABEL2IDX)
        n_unknown = (~known_mask).sum()
        if n_unknown:
            warnings.warn(f"[{split}] {n_unknown} rows with unknown emotion label — skipping.")
        df = df[known_mask].reset_index(drop=True)

        self.filepaths = df["filepath"].tolist()
        self.labels    = df["Emotion"].map(LABEL2IDX).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        waveform = load_audio(self.filepaths[idx])
        features = extract_features(waveform)
        return features, self.labels[idx]


# Quick sanity-check
train_ds = MELDAudioDataset("train")
dev_ds   = MELDAudioDataset("dev")
print(f"Train: {len(train_ds)} | Dev: {len(dev_ds)}")
x, y = train_ds[0]
print(f"Feature shape: {x.shape}  |  Label: {y} ({LABELS[y]})")

C:\Users\Andrei\AppData\Local\Temp\ipykernel_28000\3160411227.py:26: UserWarning: [train] 1 audio files not found — skipping.
  warnings.warn(f"[{split}] {n_missing} audio files not found — skipping.")
C:\Users\Andrei\AppData\Local\Temp\ipykernel_28000\3160411227.py:26: UserWarning: [dev] 1 audio files not found — skipping.
  warnings.warn(f"[{split}] {n_missing} audio files not found — skipping.")


Train: 9988 | Dev: 1108


e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Feature shape: torch.Size([64, 601])  |  Label: 6 (neutral)


## 4. Class-Balanced Data Loader

In [4]:
def make_weighted_sampler(dataset: MELDAudioDataset) -> WeightedRandomSampler:
    """Return a WeightedRandomSampler that up-samples minority classes."""
    counts = Counter(dataset.labels)
    # Per-sample weight = 1 / sqrt(class_count)  (softer than 1/count)
    sample_weights = [1.0 / (counts[lbl] ** 0.5) for lbl in dataset.labels]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )


def make_class_weights(dataset: MELDAudioDataset, device) -> torch.Tensor:
    """Return per-class loss weights (normalised inverse-sqrt frequency)."""
    counts = Counter(dataset.labels)
    counts_tensor = torch.tensor(
        [counts[i] for i in range(NUM_CLASSES)], dtype=torch.float32
    )
    weights = 1.0 / counts_tensor.sqrt()
    weights = weights / weights.mean()   # average weight ≈ 1
    return weights.to(device)


sampler      = make_weighted_sampler(train_ds)
class_weights = make_class_weights(train_ds, DEVICE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=0)
dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_SIZE, shuffle=False,    num_workers=0)

print("Class weights:", class_weights.cpu().numpy().round(3))

Class weights: [0.816 1.652 1.661 0.651 1.04  0.783 0.396]


## 5. Model

In [5]:
class AudioCNN(nn.Module):
    """
    4-layer CNN with:
    - Batch normalisation after every conv block for training stability
    - Adaptive average pooling so the classifier head is input-size agnostic
    - Dropout before the final linear layer
    """

    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.encoder = nn.Sequential(
            conv_block(1,   32),
            conv_block(32,  64),
            conv_block(64,  128),
            conv_block(128, 256),
        )

        # Collapses spatial dims to 1×1 regardless of input length
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout     = nn.Dropout(0.4)
        self.classifier  = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, n_mels, time]  →  add channel dim
        x = x.unsqueeze(1)          # [B, 1, n_mels, time]
        x = self.encoder(x)         # [B, 256, H', W']
        x = self.global_pool(x)     # [B, 256, 1, 1]
        x = x.flatten(1)            # [B, 256]
        x = self.dropout(x)
        return self.classifier(x)   # [B, num_classes]


# Shape smoke-test
_model = AudioCNN()
_x     = torch.randn(2, 64, 601)   # (batch=2, n_mels=64, time=601)
_out   = _model(_x)
assert _out.shape == (2, NUM_CLASSES), f"Unexpected output shape: {_out.shape}"
print(f"Model output shape: {_out.shape}  ✓")
del _model, _x, _out

Model output shape: torch.Size([2, 7])  ✓


## 6. Training Utilities

In [6]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device,
) -> float:
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device,
):
    """Returns (accuracy, macro_f1, classification_report_str)."""
    model.eval()
    all_preds, all_labels = [], []
    for x, y in loader:
        preds = model(x.to(device)).argmax(dim=1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())

    acc      = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    report   = classification_report(
        all_labels, all_preds, target_names=LABELS, digits=4, zero_division=0
    )
    return acc, macro_f1, report

## 7. Train

In [7]:
model     = AudioCNN().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5, verbose=True
)
criterion = nn.CrossEntropyLoss(weight=class_weights)

best_f1       = 0.0
patience_left = PATIENCE
save_path     = MODEL_DIR / "best_audio_model.pth"

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_acc, val_f1, val_report = evaluate(model, dev_loader, DEVICE)

    scheduler.step(val_f1)

    improved = val_f1 > best_f1
    if improved:
        best_f1 = val_f1
        patience_left = PATIENCE
        torch.save(model.state_dict(), save_path)
        flag = "  ✓ saved"
    else:
        patience_left -= 1
        flag = f"  (patience {patience_left}/{PATIENCE})"

    print(
        f"Epoch {epoch:3d}/{EPOCHS}  "
        f"loss={train_loss:.4f}  "
        f"val_acc={val_acc:.4f}  "
        f"val_macro_f1={val_f1:.4f}{flag}"
    )

    if patience_left == 0:
        print(f"\nEarly stopping triggered after {epoch} epochs.")
        break

print(f"\nBest macro-F1: {best_f1:.4f}  →  model saved to {save_path}")

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2+cu118
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch   1/50  loss=1.9450  val_acc=0.2139  val_macro_f1=0.1478  ✓ saved
Epoch   2/50  loss=1.9253  val_acc=0.3005  val_macro_f1=0.1926  ✓ saved
Epoch   3/50  loss=1.9255  val_acc=0.2374  val_macro_f1=0.1759  (patience 9/10)
Epoch   4/50  loss=1.9142  val_acc=0.1796  val_macro_f1=0.1357  (patience 8/10)
Epoch   5/50  loss=1.9168  val_acc=0.1968  val_macro_f1=0.1668  (patience 7/10)
Epoch   6/50  loss=1.9127  val_acc=0.2310  val_macro_f1=0.1799  (patience 6/10)
Epoch   7/50  loss=1.9067  val_acc=0.3366  val_macro_f1=0.1927  ✓ saved
Epoch   8/50  loss=1.9043  val_acc=0.1859  val_macro_f1=0.1151  (patience 9/10)
Epoch   9/50  loss=1.9023  val_acc=0.2256  val_macro_f1=0.1762  (patience 8/10)
Epoch  10/50  loss=1.8970  val_acc=0.3222  val_macro_f1=0.1936  ✓ saved
Epoch  11/50  loss=1.8954  val_acc=0.2085  val_macro_f1=0.1745  (patience 9/10)
Epoch  12/50  loss=1.8945  val_acc=0.3132  val_macro_f1=0.2128  ✓ saved
Epoch  13/50  loss=1.8878  val_acc=0.2247  val_macro_f1=0.1762  (patience 9/10)


## 8. Final Evaluation

In [ ]:
# Load the best checkpoint and evaluate on the dev set
best_model = AudioCNN().to(DEVICE)
best_model.load_state_dict(torch.load(save_path, map_location=DEVICE))

acc, macro_f1, report = evaluate(best_model, dev_loader, DEVICE)
print(f"Dev accuracy : {acc:.4f}")
print(f"Dev macro-F1 : {macro_f1:.4f}")
print()
print(report)

Dev accuracy : 0.3105
Dev macro-F1 : 0.2168

              precision    recall  f1-score   support

       anger     0.3155    0.3856    0.3471       153
     disgust     0.0000    0.0000    0.0000        22
        fear     0.0870    0.0500    0.0635        40
   happiness     0.2048    0.1043    0.1382       163
     sadness     0.1891    0.4685    0.2694       111
    surprise     0.2231    0.3600    0.2755       150
     neutral     0.5594    0.3412    0.4238       469

    accuracy                         0.3105      1108
   macro avg     0.2256    0.2442    0.2168      1108
weighted avg     0.3628    0.3105    0.3142      1108



: 